# Notebook 00 — Multi-Seed DRY-RUN (Patch Inspection)

## Purpose

**READ-ONLY.** Validates the patches the multi-seed runner will apply.

For each of the 15 pipeline notebooks:
1. Load notebook from disk
2. Apply seed + path patches in memory (DO NOT save back to original)
3. Write the patched copy to `/content/dry_run_patched/seed{N}/{notebook_name}`
4. Report what changed: which cells, which lines, what values

## What this notebook does NOT do

- Does NOT modify any original notebooks
- Does NOT execute any pipeline code
- Does NOT create model or calibrator outputs
- Does NOT touch git

Output: a patched-notebook tree on Colab disk (not Drive) for inspection,
plus a CSV report of all patches.

## Inspect-then-decide workflow

1. Run this notebook
2. Open one or two patched notebooks in `/content/dry_run_patched/seed123/` and verify:
   - `SEED = 123` (not 42) at top
   - Output paths point to `seed123/` subdirs
   - Hardcoded `random_state=42` are now `random_state=SEED` (for UNSW + CIC training only)
3. If patches look right, the next notebook (00_multi_seed_runner.ipynb) will apply the same patches AND execute them
4. If patches look wrong, we fix the dry-run logic first


In [1]:
# Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

# Dry-run output goes to Colab local disk, NOT Drive — we don't want to clutter Drive with temp files
DRY_RUN_DIR = '/content/dry_run_patched'
os.makedirs(DRY_RUN_DIR, exist_ok=True)
print(f'Repo: {os.getcwd()}')
print(f'Dry-run output: {DRY_RUN_DIR}')

Mounted at /content/drive
Repo: /content/drive/MyDrive/XIDS_Research/xids-research
Dry-run output: /content/dry_run_patched


In [2]:
import nbformat as nbf
import pandas as pd
import re, copy, json, shutil
from pathlib import Path
from datetime import datetime

# Seeds to test the patches against
SEEDS_TO_TEST = [123, 456, 789]

# Pipeline notebook list, in execution order
PIPELINE = [
    # (notebook, dataset_tag_if_any, patch_strategy, output_path_redirects)
    ('02_train_models_v2.ipynb',          'nsl_kdd_v2',     'simple_seed',          ['MODELS_DIR', 'PREDS_DIR', 'TABLES_DIR']),
    ('02_unsw_train_models_v2.ipynb',     'unsw_nb15_v2',   'inject_seed_and_regex', ['MODELS_DIR', 'PREDS_DIR', 'PROBS_DIR', 'TABLES_DIR']),
    ('02_cic_train_models_v2.ipynb',      'cic_ids2017_v2', 'inject_seed_and_regex', ['MODELS_DIR', 'PREDS_DIR', 'PROBS_DIR', 'TABLES_DIR']),
    ('03_nsl_calibration_v2.ipynb',       'nsl_kdd_v2',     'simple_seed',          ['CALIB_OUT_DIR', 'TABLES_DIR']),
    ('03_unsw_calibration_v2.ipynb',      'unsw_nb15_v2',   'simple_seed',          ['CALIB_OUT_DIR', 'TABLES_DIR']),
    ('03_cic_calibration_v2.ipynb',       'cic_ids2017_v2', 'simple_seed',          ['CALIB_OUT_DIR', 'TABLES_DIR']),
    ('03e_refit_hybrid_calibrators.ipynb','all',            'no_seed',              ['cal_dir', 'CAL_DIR', 'TABLES_DIR']),
    ('04c_shap_canonical.ipynb',          'all',            'simple_seed',          ['SHAP_DIR', 'TABLES_DIR']),
    ('05c_stability_canonical.ipynb',     'all',            'simple_seed',          ['TABLES_DIR', 'FIGURES_DIR']),
    ('06_krishna_agreement_v3.ipynb',     'all',            'simple_seed',          ['TABLES_DIR']),
    ('07c_scts_canonical_mondrian_v2.ipynb','all',          'simple_seed',          ['TABLES_DIR']),
    ('07d_scts_calib_health.ipynb',       'all',            'no_seed',              ['TABLES_DIR']),
    ('07e_phase_a_strict_protocol.ipynb', 'all',            'simple_seed_keep_boot',['CAL_DIR', 'TABLES_DIR']),
    ('07f_phase_a_diagnostic.ipynb',      'all',            'simple_seed',          ['TABLES_DIR']),
    ('08_bootstrap_cis.ipynb',            'all',            'simple_seed_keep_defaults', ['TABLES_DIR']),
]

print(f'Pipeline: {len(PIPELINE)} notebooks')
print(f'Seeds to dry-run: {SEEDS_TO_TEST}')

Pipeline: 15 notebooks
Seeds to dry-run: [123, 456, 789]


In [3]:
# Patch primitives.
# All return (new_source, n_changes, change_log)

def patch_simple_seed(source: str, new_seed: int) -> tuple:
    """Replace 'SEED = 42' (top-level) with 'SEED = <new_seed>'."""
    pattern = re.compile(r'^(\s*SEED\s*=\s*)42(\s*(?:#.*)?)$', re.MULTILINE)
    log = []
    n = 0
    def repl(m):
        nonlocal n
        n += 1
        log.append(f'SEED = 42 -> SEED = {new_seed}')
        return f'{m.group(1)}{new_seed}{m.group(2)}'
    new_src = pattern.sub(repl, source)
    return new_src, n, log

def patch_inject_seed_and_regex(source: str, new_seed: int) -> tuple:
    """For UNSW/CIC training notebooks: no top-level SEED constant.
    Inject SEED + seed-init at the start of the source, then regex-replace
    every `random_state=42` with `random_state=SEED`.

    Also handles the case where a hardcoded 42 is used in train_test_split.
    """
    log = []

    # Check if there's already a SEED = ... line. If yes, swap it via simple_seed.
    if re.search(r'^\s*SEED\s*=\s*\d+', source, re.MULTILINE):
        src1, n1, lg1 = patch_simple_seed(source, new_seed)
        log.extend(lg1)
    else:
        # No SEED constant. We'll inject it via the path-redirect cell instead (see add_path_override_cell).
        # Here we just do the regex replacement.
        src1, n1 = source, 0

    # Regex-replace random_state=42
    pattern_rs = re.compile(r'random_state\s*=\s*42\b')
    matches_rs = pattern_rs.findall(src1)
    n_rs = len(matches_rs)
    src2 = pattern_rs.sub('random_state=SEED', src1)
    if n_rs > 0:
        log.append(f'{n_rs}x random_state=42 -> random_state=SEED')

    return src2, n1 + n_rs, log

def patch_simple_seed_keep_boot(source: str, new_seed: int) -> tuple:
    """For 07e: swap SEED but leave BOOTSTRAP_SEED at 42."""
    src, n, log = patch_simple_seed(source, new_seed)
    # Verify BOOTSTRAP_SEED is unchanged (it has a different variable name so simple_seed shouldn't touch it,
    # but we explicitly check)
    if 'BOOTSTRAP_SEED = 42' in src:
        log.append('BOOTSTRAP_SEED = 42 preserved (intentional)')
    return src, n, log

def patch_simple_seed_keep_defaults(source: str, new_seed: int) -> tuple:
    """For 08: swap top-level SEED but leave function default seed=42 untouched.
    The function defaults are bootstrap-resampling RNG seeds, which must stay constant
    across seed runs so the resampling procedure is identical.
    """
    src, n, log = patch_simple_seed(source, new_seed)
    # Verify function defaults (e.g., `seed=42` in def signature) untouched
    if re.search(r'def\s+\w+\([^)]*seed\s*=\s*42', src):
        log.append('Function-default seed=42 preserved (intentional)')
    return src, n, log

def patch_no_seed(source: str, new_seed: int) -> tuple:
    return source, 0, ['(no seed in this notebook)']

PATCH_FUNCTIONS = {
    'simple_seed': patch_simple_seed,
    'inject_seed_and_regex': patch_inject_seed_and_regex,
    'simple_seed_keep_boot': patch_simple_seed_keep_boot,
    'simple_seed_keep_defaults': patch_simple_seed_keep_defaults,
    'no_seed': patch_no_seed,
}

print('Patch functions defined:', list(PATCH_FUNCTIONS.keys()))

Patch functions defined: ['simple_seed', 'inject_seed_and_regex', 'simple_seed_keep_boot', 'simple_seed_keep_defaults', 'no_seed']


In [4]:
# Build the path-override cell that gets injected into each patched notebook.
# This cell defines SEED (for notebooks that don't have it) and redirects all
# output paths to seed-suffixed directories.

def build_path_override_cell(notebook_name: str, dataset_tag: str, new_seed: int, redirects: list) -> str:
    """Returns the Python source for a cell that overrides all path constants.

    The cell is meant to be injected AFTER the notebook's original constants cell so it
    overrides constants defined there.
    """
    REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

    lines = [
        '# === MULTI-SEED PATCH: path overrides for seed=' + str(new_seed) + ' ===',
        'from pathlib import Path as _P',
        '_REPO = "' + REPO + '"',
        '_SEED_TAG = "seed' + str(new_seed) + '"',
        '',
        '# Ensure SEED variable is set (some notebooks define it; this is a backstop)',
        'try:',
        '    SEED',
        'except NameError:',
        '    SEED = ' + str(new_seed),
        '',
    ]

    # Per-notebook path overrides
    if notebook_name == '02_train_models_v2.ipynb':
        lines.extend([
            'MODELS_DIR = _P(_REPO) / "models" / "nsl_kdd_v2" / _SEED_TAG',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '02_unsw_train_models_v2.ipynb':
        lines.extend([
            'MODELS_DIR = _P(_REPO) / "models" / "unsw_nb15_v2" / _SEED_TAG',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name == '02_cic_train_models_v2.ipynb':
        lines.extend([
            'MODELS_DIR = _P(_REPO) / "models" / "cic_ids2017_v2" / _SEED_TAG',
            'PREDS_DIR = MODELS_DIR / "predictions"',
            'PROBS_DIR = MODELS_DIR / "probabilities"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:',
            '    _d.mkdir(parents=True, exist_ok=True)',
        ])
    elif notebook_name.startswith('03_') and 'calibration' in notebook_name:
        ds = dataset_tag
        lines.extend([
            f'CALIB_OUT_DIR = _P(_REPO) / "calibrators" / "{ds}" / _SEED_TAG',
            f'_MODELS_SEED_DIR = _P(_REPO) / "models" / "{ds}" / _SEED_TAG',
            f'_MODELS_SEED_PROBS = _MODELS_SEED_DIR / "probabilities"',
            f'_MODELS_SEED_PREDS = _MODELS_SEED_DIR / "predictions"',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            '# Calibration notebooks read model probabilities — these now come from seed-suffixed dirs',
            'try:',
            '    PROBS_DIR = _MODELS_SEED_PROBS',
            '    PREDS_DIR = _MODELS_SEED_PREDS',
            'except Exception: pass',
        ])
    elif notebook_name == '03e_refit_hybrid_calibrators.ipynb':
        lines.extend([
            'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
            '    (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
            '# 03e reads from calibrators/{ds}/seed{N}/ and writes back there',
        ])
    elif notebook_name == '04c_shap_canonical.ipynb':
        lines.extend([
            'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
            '    (_P(_REPO) / "shap_values" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
        ])
    else:
        # Generic redirect for stages 05c, 06, 07*, 08
        lines.extend([
            'TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG',
            'FIGURES_DIR = _P(_REPO) / "results" / "figures" / _SEED_TAG',
            'TABLES_DIR.mkdir(parents=True, exist_ok=True)',
            'FIGURES_DIR.mkdir(parents=True, exist_ok=True)',
        ])
        if dataset_tag == 'all' or notebook_name.startswith('07'):
            lines.extend([
                '# 07e refits calibrators; needs seed-suffixed calibrator dirs',
                'for _ds in ["nsl_kdd_v2", "unsw_nb15_v2", "cic_ids2017_v2"]:',
                '    (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)',
            ])

    lines.append('print(f"[multi-seed patch] SEED={SEED}, TABLES_DIR={TABLES_DIR}")')
    return '\n'.join(lines)

# Test it
print(build_path_override_cell('02_train_models_v2.ipynb', 'nsl_kdd_v2', 123, []))
print()
print('---')
print(build_path_override_cell('03_unsw_calibration_v2.ipynb', 'unsw_nb15_v2', 123, []))

# === MULTI-SEED PATCH: path overrides for seed=123 ===
from pathlib import Path as _P
_REPO = "/content/drive/MyDrive/XIDS_Research/xids-research"
_SEED_TAG = "seed123"

# Ensure SEED variable is set (some notebooks define it; this is a backstop)
try:
    SEED
except NameError:
    SEED = 123

MODELS_DIR = _P(_REPO) / "models" / "nsl_kdd_v2" / _SEED_TAG
PREDS_DIR = MODELS_DIR / "predictions"
PROBS_DIR = MODELS_DIR / "probabilities"
TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG
for _d in [MODELS_DIR, PREDS_DIR, PROBS_DIR, TABLES_DIR]:
    _d.mkdir(parents=True, exist_ok=True)
print(f"[multi-seed patch] SEED={SEED}, TABLES_DIR={TABLES_DIR}")

---
# === MULTI-SEED PATCH: path overrides for seed=123 ===
from pathlib import Path as _P
_REPO = "/content/drive/MyDrive/XIDS_Research/xids-research"
_SEED_TAG = "seed123"

# Ensure SEED variable is set (some notebooks define it; this is a backstop)
try:
    SEED
except NameError:
    SEED = 123

CALIB_OUT_DIR = _P(_REPO) / "calibrato

In [5]:
# Main dry-run loop
# For each seed, for each notebook:
#   1. Load original notebook
#   2. Apply seed patch
#   3. Inject path-override cell
#   4. Write patched notebook to /content/dry_run_patched/seed{N}/
#   5. Record what changed

import nbformat as nbf

nb_dir_orig = Path(REPO) / 'notebooks'
all_patch_records = []
all_missing = []

for new_seed in SEEDS_TO_TEST:
    seed_dir = Path(DRY_RUN_DIR) / f'seed{new_seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)

    print(f'\n{"="*70}')
    print(f'DRY-RUN: SEED = {new_seed}')
    print(f'{"="*70}')

    for nb_name, ds_tag, strategy, redirects in PIPELINE:
        src_path = nb_dir_orig / nb_name
        if not src_path.exists():
            all_missing.append((new_seed, nb_name))
            print(f'  [MISSING] {nb_name}')
            continue

        # Load
        nb_obj = nbf.read(str(src_path), as_version=4)

        # Patch each code cell
        patch_fn = PATCH_FUNCTIONS[strategy]
        total_changes = 0
        change_log_combined = []
        for cell in nb_obj.cells:
            if cell.cell_type != 'code':
                continue
            new_source, n_changes, log = patch_fn(cell.source, new_seed)
            cell.source = new_source
            total_changes += n_changes
            change_log_combined.extend(log)

        # Inject the path-override cell after the FIRST code cell that defines constants
        # (typically cell index 2 or 3 — the imports + constants cell)
        override_code = build_path_override_cell(nb_name, ds_tag, new_seed, redirects)
        override_cell = nbf.v4.new_code_cell(override_code)

        # Find the constants cell (containing SEED= or first import statement)
        # We insert override AFTER it
        insertion_idx = None
        for idx, cell in enumerate(nb_obj.cells):
            if cell.cell_type != 'code':
                continue
            # Insert after first cell that contains SEED= or imports
            if ('SEED' in cell.source) or ('import numpy' in cell.source) or ('import pandas' in cell.source):
                insertion_idx = idx + 1
                break
        if insertion_idx is None:
            # Fallback: insert at position 1 (just after first code cell)
            for idx, cell in enumerate(nb_obj.cells):
                if cell.cell_type == 'code':
                    insertion_idx = idx + 1
                    break

        if insertion_idx is not None:
            nb_obj.cells.insert(insertion_idx, override_cell)

        # Write the patched notebook to dry-run dir
        out_path = seed_dir / nb_name
        with open(out_path, 'w') as f:
            nbf.write(nb_obj, f)

        # Record
        all_patch_records.append({
            'seed': new_seed,
            'notebook': nb_name,
            'strategy': strategy,
            'n_changes': total_changes,
            'override_cell_inserted_at': insertion_idx,
            'change_log': '; '.join(change_log_combined) if change_log_combined else '(none)',
        })

        print(f'  {nb_name:<42} strategy={strategy:<28} changes={total_changes} override@cell{insertion_idx}')

print(f'\n\nTotal patches recorded: {len(all_patch_records)}')
print(f'Missing notebooks: {len(all_missing)}')


DRY-RUN: SEED = 123
  02_train_models_v2.ipynb                   strategy=simple_seed                  changes=1 override@cell4
  02_unsw_train_models_v2.ipynb              strategy=inject_seed_and_regex        changes=8 override@cell6
  02_cic_train_models_v2.ipynb               strategy=inject_seed_and_regex        changes=8 override@cell6
  03_nsl_calibration_v2.ipynb                strategy=simple_seed                  changes=1 override@cell4
  03_unsw_calibration_v2.ipynb               strategy=simple_seed                  changes=1 override@cell4
  03_cic_calibration_v2.ipynb                strategy=simple_seed                  changes=1 override@cell4
  03e_refit_hybrid_calibrators.ipynb         strategy=no_seed                      changes=0 override@cell4
  04c_shap_canonical.ipynb                   strategy=simple_seed                  changes=1 override@cell4
  05c_stability_canonical.ipynb              strategy=simple_seed                  changes=1 override@cell4
  06_kr

In [6]:
# Show full change log per (seed, notebook)
df_patches = pd.DataFrame(all_patch_records)

print('=' * 90)
print('DETAILED PATCH SUMMARY (seed=123 only — same pattern for 456 and 789)')
print('=' * 90)

for _, row in df_patches[df_patches['seed'] == 123].iterrows():
    print(f'\n--- {row["notebook"]} (strategy: {row["strategy"]}) ---')
    print(f'  Override cell inserted at: index {row["override_cell_inserted_at"]}')
    print(f'  Total changes: {row["n_changes"]}')
    print(f'  Log: {row["change_log"]}')

DETAILED PATCH SUMMARY (seed=123 only — same pattern for 456 and 789)

--- 02_train_models_v2.ipynb (strategy: simple_seed) ---
  Override cell inserted at: index 4
  Total changes: 1
  Log: SEED = 42 -> SEED = 123

--- 02_unsw_train_models_v2.ipynb (strategy: inject_seed_and_regex) ---
  Override cell inserted at: index 6
  Total changes: 8
  Log: 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 2x random_state=42 -> random_state=SEED

--- 02_cic_train_models_v2.ipynb (strategy: inject_seed_and_regex) ---
  Override cell inserted at: index 6
  Total changes: 8
  Log: 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> random_state=SEED; 1x random_state=42 -> r

In [7]:
# For each patched notebook (seed=123), spot-check the changes
# Show: first occurrence of SEED= in the patched file, first 5 random_state= references,
# and that the override cell got inserted

print('=' * 90)
print('PATCH SPOT-CHECK — verify each patched seed=123 notebook')
print('=' * 90)

for nb_name, ds_tag, strategy, redirects in PIPELINE:
    patched_path = Path(DRY_RUN_DIR) / 'seed123' / nb_name
    if not patched_path.exists():
        print(f'\n--- {nb_name}: MISSING ---')
        continue

    nb_obj = nbf.read(str(patched_path), as_version=4)
    print(f'\n--- {nb_name} ---')

    found_seed_line = None
    found_random_state_lines = []
    found_override_cell = False

    for ci, cell in enumerate(nb_obj.cells):
        if cell.cell_type != 'code':
            continue
        if 'MULTI-SEED PATCH' in cell.source:
            found_override_cell = True
            # Print first 2 lines of the override
            for line in cell.source.split('\n')[:3]:
                print(f'  [override @ cell {ci}] {line[:80]}')

        for line in cell.source.split('\n'):
            if found_seed_line is None and re.search(r'^\s*SEED\s*=\s*\d+', line):
                found_seed_line = (ci, line.strip())
            for m in re.finditer(r'random_state\s*=\s*\S+', line):
                if len(found_random_state_lines) < 3:
                    found_random_state_lines.append((ci, m.group(0)))

    if found_seed_line:
        print(f'  SEED line: cell {found_seed_line[0]}: {found_seed_line[1]}')
    if found_random_state_lines:
        print(f'  random_state samples:')
        for ci, val in found_random_state_lines:
            print(f'    cell {ci}: {val}')
    if not found_override_cell:
        print(f'  WARNING: no override cell found!')

PATCH SPOT-CHECK — verify each patched seed=123 notebook

--- 02_train_models_v2.ipynb ---
  [override @ cell 4] # === MULTI-SEED PATCH: path overrides for seed=123 ===
  [override @ cell 4] from pathlib import Path as _P
  [override @ cell 4] _REPO = "/content/drive/MyDrive/XIDS_Research/xids-research"
  SEED line: cell 3: SEED = 123
  random_state samples:
    cell 7: random_state=SEED)
    cell 7: random_state=SEED,
    cell 13: random_state=SEED,

--- 02_unsw_train_models_v2.ipynb ---
  [override @ cell 6] # === MULTI-SEED PATCH: path overrides for seed=123 ===
  [override @ cell 6] from pathlib import Path as _P
  [override @ cell 6] _REPO = "/content/drive/MyDrive/XIDS_Research/xids-research"
  SEED line: cell 6: SEED = 123
  random_state samples:
    cell 10: random_state=SEED,
    cell 12: random_state=SEED,
    cell 14: random_state=SEED,

--- 02_cic_train_models_v2.ipynb ---
  [override @ cell 6] # === MULTI-SEED PATCH: path overrides for seed=123 ===
  [override @ cell 6] fr

In [8]:
# Save the patch report to /content/dry_run_patched/ for review
out_csv = Path(DRY_RUN_DIR) / 'patch_report.csv'
df_patches.to_csv(out_csv, index=False)

# Also write a short summary
summary_lines = [
    f'# Multi-Seed Dry-Run Patch Report',
    f'',
    f'Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'Seeds tested: {SEEDS_TO_TEST}',
    f'Pipeline notebooks: {len(PIPELINE)}',
    f'',
    f'## Patches applied per seed',
    f'',
]
for s in SEEDS_TO_TEST:
    sub = df_patches[df_patches['seed'] == s]
    total = sub['n_changes'].sum()
    summary_lines.append(f'- Seed {s}: {len(sub)} notebooks patched, {total} total changes')

summary_lines.extend([
    '',
    '## Strategy counts',
    '',
])
strat_counts = df_patches[df_patches['seed'] == SEEDS_TO_TEST[0]]['strategy'].value_counts()
for k, v in strat_counts.items():
    summary_lines.append(f'- {k}: {v} notebooks')

summary_lines.extend([
    '',
    '## Output location',
    '',
    f'Patched notebooks (read-only inspection): {DRY_RUN_DIR}/seed{{N}}/',
    '',
    '## Next step',
    '',
    'Inspect a couple of patched notebooks manually. Specifically check:',
    '1. `seed123/02_train_models_v2.ipynb` cell 3 — should show `SEED = 123`',
    '2. `seed123/02_unsw_train_models_v2.ipynb` cell 9 — should show `random_state=SEED`',
    '3. `seed123/03_nsl_calibration_v2.ipynb` cell 3 — should show `SEED = 123`',
    '4. `seed123/07e_phase_a_strict_protocol.ipynb` — `SEED = 123` AND `BOOTSTRAP_SEED = 42` preserved',
    '5. `seed123/08_bootstrap_cis.ipynb` — `SEED = 123` AND `def bootstrap_ci(..., seed=42)` defaults preserved',
    '6. Every notebook should have a `MULTI-SEED PATCH:` cell injected near the top',
    '',
    'If everything looks right, proceed to the multi-seed runner notebook.',
])

summary_md = '\n'.join(summary_lines)
with open(Path(DRY_RUN_DIR) / 'README.md', 'w') as f:
    f.write(summary_md)

print(f'Saved: {out_csv}')
print(f'Saved: {Path(DRY_RUN_DIR) / "README.md"}')
print()
print(summary_md)

Saved: /content/dry_run_patched/patch_report.csv
Saved: /content/dry_run_patched/README.md

# Multi-Seed Dry-Run Patch Report

Date: 2026-06-09 05:55
Seeds tested: [123, 456, 789]
Pipeline notebooks: 15

## Patches applied per seed

- Seed 123: 15 notebooks patched, 28 total changes
- Seed 456: 15 notebooks patched, 28 total changes
- Seed 789: 15 notebooks patched, 28 total changes

## Strategy counts

- simple_seed: 9 notebooks
- inject_seed_and_regex: 2 notebooks
- no_seed: 2 notebooks
- simple_seed_keep_boot: 1 notebooks
- simple_seed_keep_defaults: 1 notebooks

## Output location

Patched notebooks (read-only inspection): /content/dry_run_patched/seed{N}/

## Next step

Inspect a couple of patched notebooks manually. Specifically check:
1. `seed123/02_train_models_v2.ipynb` cell 3 — should show `SEED = 123`
2. `seed123/02_unsw_train_models_v2.ipynb` cell 9 — should show `random_state=SEED`
3. `seed123/03_nsl_calibration_v2.ipynb` cell 3 — should show `SEED = 123`
4. `seed123/07e_p

In [9]:
import nbformat as nbf
import re
from pathlib import Path
import pandas as pd

DRY_RUN_DIR = Path('/content/dry_run_patched')
SEEDS = [123, 456, 789]

# Expected pipeline notebooks (must all exist in dry-run output)
EXPECTED_NOTEBOOKS = [
    '02_train_models_v2.ipynb',
    '02_unsw_train_models_v2.ipynb',
    '02_cic_train_models_v2.ipynb',
    '03_nsl_calibration_v2.ipynb',
    '03_unsw_calibration_v2.ipynb',
    '03_cic_calibration_v2.ipynb',
    '03e_refit_hybrid_calibrators.ipynb',
    '04c_shap_canonical.ipynb',
    '05c_stability_canonical.ipynb',
    '06_krishna_agreement_v3.ipynb',
    '07c_scts_canonical_mondrian_v2.ipynb',
    '07d_scts_calib_health.ipynb',
    '07e_phase_a_strict_protocol.ipynb',
    '07f_phase_a_diagnostic.ipynb',
    '08_bootstrap_cis.ipynb',
]

# Notebooks that should preserve certain literals (intentional non-patches)
PRESERVE_BOOTSTRAP_SEED = ['07e_phase_a_strict_protocol.ipynb']  # BOOTSTRAP_SEED = 42 stays
PRESERVE_FUNCTION_DEFAULTS = ['08_bootstrap_cis.ipynb']  # def f(..., seed=42) stays

# Notebooks that shouldn't have ANY SEED variable (deterministic, no RNG)
NO_SEED_EXPECTED = ['03e_refit_hybrid_calibrators.ipynb', '07d_scts_calib_health.ipynb']

def get_all_code_text(nb_obj):
    """Concatenate all code cells into a single string for searching."""
    return '\n\n'.join(c.source for c in nb_obj.cells if c.cell_type == 'code')

def has_override_cell(nb_obj):
    """Check if the multi-seed patch override cell was injected."""
    for c in nb_obj.cells:
        if c.cell_type == 'code' and 'MULTI-SEED PATCH' in c.source:
            return True
    return False

def get_seed_assignments(text):
    """Find every line that assigns SEED = <number>."""
    return re.findall(r'^\s*SEED\s*=\s*(\d+)', text, re.MULTILINE)

def count_random_state_literal_42(text):
    """Count remaining `random_state=42` literals (should be 0 after patching)."""
    return len(re.findall(r'random_state\s*=\s*42\b', text))

def count_random_state_var(text):
    """Count `random_state=SEED` references (should be > 0 in training notebooks)."""
    return len(re.findall(r'random_state\s*=\s*SEED\b', text))

def has_bootstrap_seed_42(text):
    """Check if BOOTSTRAP_SEED = 42 is preserved."""
    return bool(re.search(r'^\s*BOOTSTRAP_SEED\s*=\s*42', text, re.MULTILINE))

def has_function_default_seed_42(text):
    """Check if `def f(..., seed=42)` defaults are preserved."""
    return bool(re.search(r'def\s+\w+\([^)]*\bseed\s*=\s*42', text))

results = []

for seed in SEEDS:
    seed_dir = DRY_RUN_DIR / f'seed{seed}'
    if not seed_dir.exists():
        print(f'MISSING DIRECTORY: {seed_dir}')
        continue

    for nb_name in EXPECTED_NOTEBOOKS:
        nb_path = seed_dir / nb_name
        check = {
            'seed': seed,
            'notebook': nb_name,
            'file_exists': nb_path.exists(),
        }

        if not nb_path.exists():
            check['status'] = 'MISSING'
            results.append(check)
            continue

        nb_obj = nbf.read(str(nb_path), as_version=4)
        text = get_all_code_text(nb_obj)

        # === Check 1: override cell injected ===
        check['has_override_cell'] = has_override_cell(nb_obj)

        # === Check 2: SEED variable assignments ===
        seed_assignments = get_seed_assignments(text)
        check['seed_assignments'] = seed_assignments
        check['n_seed_assignments'] = len(seed_assignments)

        # All SEED= assignments should be the new seed value (except in override cell, which
        # uses str(new_seed) directly so it'll match anyway)
        if nb_name in NO_SEED_EXPECTED:
            # These shouldn't have SEED= except possibly in the injected override cell fallback
            check['seed_value_correct'] = all(int(v) == seed for v in seed_assignments) if seed_assignments else True
        else:
            check['seed_value_correct'] = (
                len(seed_assignments) > 0 and
                all(int(v) == seed for v in seed_assignments)
            )

        # === Check 3: no leftover `random_state=42` ===
        rs_42_count = count_random_state_literal_42(text)
        rs_seed_count = count_random_state_var(text)
        check['random_state_42_remaining'] = rs_42_count
        check['random_state_SEED_count'] = rs_seed_count
        check['no_leftover_rs_42'] = (rs_42_count == 0)

        # === Check 4: BOOTSTRAP_SEED preserved where required ===
        if nb_name in PRESERVE_BOOTSTRAP_SEED:
            check['bootstrap_seed_preserved'] = has_bootstrap_seed_42(text)
        else:
            check['bootstrap_seed_preserved'] = 'N/A'

        # === Check 5: function defaults preserved where required ===
        if nb_name in PRESERVE_FUNCTION_DEFAULTS:
            check['function_defaults_preserved'] = has_function_default_seed_42(text)
        else:
            check['function_defaults_preserved'] = 'N/A'

        # === Overall pass/fail ===
        failures = []
        if not check['has_override_cell']:
            failures.append('no override cell')
        if not check['seed_value_correct']:
            failures.append(f'wrong SEED values: {seed_assignments}')
        if not check['no_leftover_rs_42']:
            failures.append(f'{rs_42_count} random_state=42 remaining')
        if nb_name in PRESERVE_BOOTSTRAP_SEED and check['bootstrap_seed_preserved'] is not True:
            failures.append('BOOTSTRAP_SEED not preserved')
        if nb_name in PRESERVE_FUNCTION_DEFAULTS and check['function_defaults_preserved'] is not True:
            failures.append('function defaults not preserved')

        check['status'] = 'PASS' if not failures else 'FAIL: ' + '; '.join(failures)
        results.append(check)

df_results = pd.DataFrame(results)

# === Report ===
print('=' * 100)
print('AUTOMATED PATCH VERIFICATION')
print('=' * 100)

# Pass/fail summary
total = len(df_results)
n_pass = (df_results['status'] == 'PASS').sum()
n_fail = total - n_pass
n_missing = (df_results['status'] == 'MISSING').sum()

print(f'\nTotal checks: {total} ({len(SEEDS)} seeds x {len(EXPECTED_NOTEBOOKS)} notebooks)')
print(f'  PASS:    {n_pass}')
print(f'  FAIL:    {n_fail - n_missing}')
print(f'  MISSING: {n_missing}')

# Per-notebook summary
print(f'\n--- Per-notebook (across seeds 123/456/789) ---')
for nb_name in EXPECTED_NOTEBOOKS:
    sub = df_results[df_results['notebook'] == nb_name]
    statuses = sub['status'].tolist()
    all_pass = all(s == 'PASS' for s in statuses)
    marker = 'OK' if all_pass else 'FAIL'
    print(f'  [{marker}] {nb_name:<42} statuses: {statuses}')

# Detailed failures
fails = df_results[~df_results['status'].str.startswith('PASS')]
if len(fails) > 0:
    print(f'\n--- DETAILED FAILURES ({len(fails)} rows) ---')
    for _, row in fails.iterrows():
        print(f'\n  seed={row["seed"]} | {row["notebook"]}')
        print(f'    status: {row["status"]}')
        print(f'    seed_assignments: {row.get("seed_assignments", "n/a")}')
        print(f'    random_state=42 remaining: {row.get("random_state_42_remaining", "n/a")}')
        print(f'    random_state=SEED count: {row.get("random_state_SEED_count", "n/a")}')
        print(f'    BOOTSTRAP_SEED preserved: {row.get("bootstrap_seed_preserved", "n/a")}')
        print(f'    function defaults preserved: {row.get("function_defaults_preserved", "n/a")}')
        print(f'    override cell: {row.get("has_override_cell", "n/a")}')
else:
    print('\nAll checks passed. Safe to proceed to runner.')

# Cross-seed consistency check: patches at different seeds should produce identical structure
# except for the seed value itself
print(f'\n--- Cross-seed consistency ---')
for nb_name in EXPECTED_NOTEBOOKS:
    s123 = df_results[(df_results['seed'] == 123) & (df_results['notebook'] == nb_name)]
    s456 = df_results[(df_results['seed'] == 456) & (df_results['notebook'] == nb_name)]
    s789 = df_results[(df_results['seed'] == 789) & (df_results['notebook'] == nb_name)]
    if len(s123) == 0 or len(s456) == 0 or len(s789) == 0:
        continue
    # Check that random_state=SEED count is identical across seeds
    rs_counts = [s123['random_state_SEED_count'].iloc[0],
                 s456['random_state_SEED_count'].iloc[0],
                 s789['random_state_SEED_count'].iloc[0]]
    if len(set(rs_counts)) > 1:
        print(f'  INCONSISTENT: {nb_name} has different random_state=SEED counts: {rs_counts}')

# Specific high-risk spot checks (paste-friendly)
print(f'\n--- High-risk spot checks (seed=123) ---')
high_risk = ['02_unsw_train_models_v2.ipynb', '02_cic_train_models_v2.ipynb',
             '07e_phase_a_strict_protocol.ipynb', '08_bootstrap_cis.ipynb']
for nb_name in high_risk:
    nb_path = DRY_RUN_DIR / 'seed123' / nb_name
    if not nb_path.exists():
        continue
    nb_obj = nbf.read(str(nb_path), as_version=4)
    text = get_all_code_text(nb_obj)
    print(f'\n  {nb_name}:')
    print(f'    SEED assignments found: {get_seed_assignments(text)}')
    print(f'    random_state=42 remaining: {count_random_state_literal_42(text)}')
    print(f'    random_state=SEED count: {count_random_state_var(text)}')
    print(f'    BOOTSTRAP_SEED = 42 present: {has_bootstrap_seed_42(text)}')
    print(f'    def f(..., seed=42) present: {has_function_default_seed_42(text)}')

# Save the report
df_results.to_csv(DRY_RUN_DIR / 'verification_report.csv', index=False)
print(f'\nSaved: {DRY_RUN_DIR / "verification_report.csv"}')

print()
print('=' * 100)
print(f'VERDICT: {"ALL PASS — proceed to runner" if n_fail == 0 else f"{n_fail} FAILURES — fix dry-run logic first"}')
print('=' * 100)

AUTOMATED PATCH VERIFICATION

Total checks: 45 (3 seeds x 15 notebooks)
  PASS:    45
  FAIL:    0
  MISSING: 0

--- Per-notebook (across seeds 123/456/789) ---
  [OK] 02_train_models_v2.ipynb                   statuses: ['PASS', 'PASS', 'PASS']
  [OK] 02_unsw_train_models_v2.ipynb              statuses: ['PASS', 'PASS', 'PASS']
  [OK] 02_cic_train_models_v2.ipynb               statuses: ['PASS', 'PASS', 'PASS']
  [OK] 03_nsl_calibration_v2.ipynb                statuses: ['PASS', 'PASS', 'PASS']
  [OK] 03_unsw_calibration_v2.ipynb               statuses: ['PASS', 'PASS', 'PASS']
  [OK] 03_cic_calibration_v2.ipynb                statuses: ['PASS', 'PASS', 'PASS']
  [OK] 03e_refit_hybrid_calibrators.ipynb         statuses: ['PASS', 'PASS', 'PASS']
  [OK] 04c_shap_canonical.ipynb                   statuses: ['PASS', 'PASS', 'PASS']
  [OK] 05c_stability_canonical.ipynb              statuses: ['PASS', 'PASS', 'PASS']
  [OK] 06_krishna_agreement_v3.ipynb              statuses: ['PASS', 'PASS

In [10]:
import nbformat as nbf
from pathlib import Path

nb = nbf.read('/content/dry_run_patched/seed123/08_bootstrap_cis.ipynb', as_version=4)
for ci, c in enumerate(nb.cells):
    if c.cell_type != 'code': continue
    for li, line in enumerate(c.source.split('\n')):
        if 'SEED' in line and '=' in line and 'BOOTSTRAP' not in line and not line.strip().startswith('#'):
            print(f'  cell {ci} line {li}: {line.rstrip()[:140]}')

  cell 3 line 8: SEED = 123
  cell 3 line 20: print(f'Bootstrap: B={B}, 95% CI, SEED={SEED}')
  cell 4 line 3: _SEED_TAG = "seed123"
  cell 4 line 9:     SEED = 123
  cell 4 line 11: TABLES_DIR = _P(_REPO) / "results" / "tables" / _SEED_TAG
  cell 4 line 12: FIGURES_DIR = _P(_REPO) / "results" / "figures" / _SEED_TAG
  cell 4 line 17:     (_P(_REPO) / "calibrators" / _ds / _SEED_TAG).mkdir(parents=True, exist_ok=True)
  cell 4 line 18: print(f"[multi-seed patch] SEED={SEED}, TABLES_DIR={TABLES_DIR}")
  cell 8 line 35:             point, lo, hi = bootstrap_ci_paired(y_c, p_c, lambda y, p: float(((p - y) ** 2).mean()), B=B, alpha=ALPHA, seed=SEED)
  cell 8 line 44:         rng = np.random.default_rng(SEED)
  cell 10 line 38:         point, lo, hi = bootstrap_ci(vals, lambda v: float(np.mean(v)), B=B, alpha=ALPHA, seed=SEED)
  cell 12 line 13:     point, lo, hi = bootstrap_ci(vals, lambda v: float(np.mean(v)), B=B, alpha=ALPHA, seed=SEED)
  cell 12 line 41:             point, lo, hi = boo

## How to inspect the patches

The patched notebooks are on Colab local disk at `/content/dry_run_patched/seed123/`.

To inspect:
1. In Colab's left sidebar, open the file browser
2. Navigate to `dry_run_patched/seed123/`
3. Open `02_train_models_v2.ipynb` (right-click -> View as JSON, or just open in a separate tab)
4. Look at cell 3 (or near top) — confirm `SEED = 123`
5. Look for an injected cell with `# === MULTI-SEED PATCH: path overrides for seed=123 ===`

Same for:
- `02_unsw_train_models_v2.ipynb` — should have multiple `random_state=SEED` lines (previously `random_state=42`)
- `07e_phase_a_strict_protocol.ipynb` — should have `SEED = 123` but `BOOTSTRAP_SEED = 42` preserved
- `08_bootstrap_cis.ipynb` — should have `SEED = 123` but function defaults `seed=42` preserved

If anything looks wrong, paste the relevant cell contents back and we'll adjust the dry-run logic.

If everything looks right, the next step is the multi-seed RUNNER notebook (which applies the same patches AND executes them via nbclient).

**Nothing has been pushed to git. Nothing on Drive has been modified. This is fully reversible.**